# 01 · Profile every file before we trust any of it

**Produces:** `data/clean/profile_files.csv` and `data/clean/profile_columns.csv`, and the first entries of our decisions log.

**Needs:** the data pack in `data/raw/`.

**Why this notebook exists:** a count is only as true as the file it came from. Before any join or any cleaning we look at every kind of file once: its shape, its data types, its missing values, its id format and its dates. We write down everything that surprised us.

## CONFIG
Everything that depends on the machine or the period lives in this one cell, so nothing else needs editing.

In [ ]:
# ---------- imports ----------
import gzip
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# show more columns when we print a dataframe
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 150)

# ---------- paths ----------
# the notebooks live in the notebooks folder, so the repo root is one folder up
if Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent
else:
    ROOT = Path.cwd()

RAW = ROOT / "data" / "raw"       # the data pack as we received it (we never edit this)
CLEAN = ROOT / "data" / "clean"   # the files our notebooks write
SLIDES = ROOT / "slides"          # charts and the excel file for the slides
CLEAN.mkdir(parents=True, exist_ok=True)
SLIDES.mkdir(parents=True, exist_ok=True)

# ---------- the two periods we compare ----------
H1_2025 = ("2025-01-01", "2025-06-30 23:59:59")   # 23:59:59 so that 30 June is included
H1_2026 = ("2026-01-01", "2026-06-30 23:59:59")

# a small helper: prints PASS or FAIL but never stops the notebook
def check(name, condition, detail=""):
    if condition:
        print("PASS", name, detail)
    else:
        print("FAIL", name, detail)

print("repo root:", ROOT)

## 1 · What is in the data pack?
We count the files and the size of every folder first.

In [ ]:
# count the files in every folder of the pack and add up their size
folders = sorted([p for p in RAW.iterdir() if p.is_dir()])

rows = []
for folder in folders:
    n_files = 0
    n_bytes = 0
    for f in folder.rglob("*"):
        if f.is_file():
            n_files = n_files + 1
            n_bytes = n_bytes + f.stat().st_size
    rows.append({"folder": folder.name, "files": n_files, "mb": round(n_bytes / 1000000, 1)})

df_folders = pd.DataFrame(rows)
print(df_folders)
print("total files:", df_folders["files"].sum())
df_folders.to_csv(CLEAN / "profile_files.csv", index=False)

### Check: is the download complete and unchanged?

In [ ]:
import hashlib

# MANIFEST.json lists every file of the pack. first check that no file is missing
manifest_path = RAW / "MANIFEST.json"
if manifest_path.exists():
    manifest = json.load(open(manifest_path))
    missing = []
    for item in manifest["files"]:
        if not (RAW / item["path"]).exists():
            missing.append(item["path"])
    check("every file in MANIFEST.json is present", len(missing) == 0, str(len(missing)) + " missing")

    # checking the hash of all files takes long, so we only spot check three files
    n_files = len(manifest["files"])
    sample_items = [manifest["files"][0], manifest["files"][n_files // 2], manifest["files"][n_files - 1]]
    all_match = True
    for item in sample_items:
        file_hash = hashlib.sha256((RAW / item["path"]).read_bytes()).hexdigest()
        if file_hash != item["sha256"]:
            all_match = False
            print("hash is different for", item["path"])
    check("spot checked SHA-256 values match the manifest", all_match)
else:
    print("no MANIFEST.json in data/raw, so we skip this check")

## 2 · A legacy booking file: read it the naive way first and watch it go wrong
The old booking system exported one csv per city per month. We pick the biggest file as our sample.

In [ ]:
legacy_files = sorted((RAW / "bookings" / "legacy").rglob("*.csv.gz"))
print("number of legacy files:", len(legacy_files))

# pick the biggest file as the sample
biggest_legacy = legacy_files[0]
for f in legacy_files:
    if f.stat().st_size > biggest_legacy.stat().st_size:
        biggest_legacy = f
print("sample file:", biggest_legacy.name, "from", biggest_legacy.parent.name)

# first try: the default way (utf-8)
try:
    df_try = pd.read_csv(biggest_legacy)
    print("read without error, shape:", df_try.shape)
except UnicodeDecodeError as err:
    print("UnicodeDecodeError:", str(err)[:100])

In [ ]:
# second try: the vendor's encoding is latin-1. we read everything as text (dtype=str) first and convert later
df_leg = pd.read_csv(biggest_legacy, encoding="latin-1", dtype=str)

print("shape:", df_leg.shape)
df_leg.head()

In [ ]:
# data types, missing values and the grain of the file
print(df_leg.dtypes)
print()
print("missing values per column:")
print(df_leg.isnull().sum())

In [ ]:
# problem 1: the vendor repeated the header line inside the file
n_repeated = (df_leg["booking_id"] == "booking_id").sum()
print("repeated header lines:", n_repeated)
df_leg = df_leg[df_leg["booking_id"] != "booking_id"]

# problem 2: spaces at the end of some booking ids (they break joins later)
n_spaces = (df_leg["booking_id"] != df_leg["booking_id"].str.strip()).sum()
print("rows with spaces around the booking id:", n_spaces)

# the grain: how many rows per booking?
n_rows = len(df_leg)
n_bookings = df_leg["booking_id"].str.strip().nunique()
print("rows:", n_rows, " bookings:", n_bookings, " rows per booking:", round(n_rows / n_bookings, 2))
print("so one row is one TEST, not one booking")

In [ ]:
# dates are written day first (dd/mm/yyyy). we must give the format, or pandas may swap day and month
print(df_leg["slot_at"].head(3).tolist())

df_leg["slot_dt"] = pd.to_datetime(df_leg["slot_at"], format="%d/%m/%Y %H:%M", errors="coerce")
print("dates that could not be read:", df_leg["slot_dt"].isnull().sum())
print("first slot:", df_leg["slot_dt"].min(), " last slot:", df_leg["slot_dt"].max())

# the values of the small columns
print(df_leg["city"].value_counts())
print(df_leg["channel"].value_counts())
print(df_leg["source"].value_counts())

# price is text for now, so make it a number to look at it
df_leg["price_num"] = pd.to_numeric(df_leg["test_price"], errors="coerce")
print(df_leg["price_num"].describe())

> **Interview angle.** Why did `pd.read_csv` need `encoding="latin-1"`, and why is guessing the encoding risky?
>
> *A good answer:* the old vendor saved the files in latin-1 and pandas assumes utf-8. If you try encodings until one works you can silently turn accented letters into different valid letters. Ask the source system what the encoding is, and write it in the loader.

## 3 · A nova booking file (json lines)
The new system writes one json object per booking, with the tests nested inside it.

In [ ]:
nova_files = sorted((RAW / "bookings" / "nova").rglob("*.jsonl.gz"))
print("number of nova files:", len(nova_files))

biggest_nova = nova_files[0]
for f in nova_files:
    if f.stat().st_size > biggest_nova.stat().st_size:
        biggest_nova = f
print("sample file:", biggest_nova.name, "from", biggest_nova.parent.name)

# read the file line by line, each line is one booking
records = []
with gzip.open(biggest_nova, "rt", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))
print("bookings in the file:", len(records))

# look at one record
print(json.dumps(records[0], indent=1))

In [ ]:
# put a few fields in a small table, to look at them the same way as the other files
ids = []
statuses = []
n_tests_list = []
channels = []
slots = []
for r in records:
    ids.append(r["booking_id"])
    statuses.append(r["status_at_export"])
    n_tests_list.append(len(r["tests"]))
    channels.append(r["channel"])
    slots.append(r["slot_at"])

df_nova_small = pd.DataFrame({"booking_id": ids, "status_at_export": statuses, "n_tests": n_tests_list,
                              "channel": channels, "slot_at": slots})
print(df_nova_small.head())
print()
print(df_nova_small["status_at_export"].value_counts())
print()
print("bookings with an empty test list:", (df_nova_small["n_tests"] == 0).sum())
print("average tests per booking:", round(df_nova_small["n_tests"].mean(), 2))
print("channel written as:", df_nova_small["channel"].unique())

# the file is named for one month, but the slot month can be different
df_nova_small["slot_month"] = df_nova_small["slot_at"].str[:7]
print()
print(df_nova_small["slot_month"].value_counts())

In [ ]:
# nova keeps the OLD booking when a visit is rescheduled, and writes a NEW booking that points back to it.
# legacy just overwrote the slot. so one moved visit is two bookings in nova.
resched_old = []
for r in records:
    if r["rescheduled_from"] is not None:
        resched_old.append(r["rescheduled_from"])

n_old_marked = (df_nova_small["status_at_export"] == "rescheduled").sum()
print("new bookings that point to an old one:", len(resched_old))
print("old bookings marked rescheduled      :", n_old_marked)
check("every rescheduled booking has a replacement", len(resched_old) == n_old_marked)

> **Interview angle.** Two systems record the same visit in different shapes. How do you make them comparable?
>
> *A good answer:* trace one real visit through both systems by hand, write down how each one shows it, write one rule that maps both to the same definition of a visit, apply it, and check the count before and after. The trap is comparing totals without the rule and calling the difference a business change.

## 4 · The other tables: collections, patients, plan, results

In [ ]:
# ----- collections (one file) -----
coll_files = sorted((RAW / "collections").glob("*.csv.gz"))
print("number of collections files:", len(coll_files))
df_coll_s = pd.read_csv(coll_files[0], dtype=str)
print(df_coll_s.shape)
print(df_coll_s.head())
print(df_coll_s["status"].value_counts())
print("exact duplicate rows (double scans):", df_coll_s.duplicated().sum())
print("missing values:")
print(df_coll_s.isnull().sum())

In [ ]:
# ----- patients -----
df_pat_s = pd.read_csv(RAW / "patients" / "patients.csv.gz", dtype=str)
print(df_pat_s.shape)
print(df_pat_s.head())
print(df_pat_s.dtypes)
print("missing values:")
print(df_pat_s.isnull().sum())
print("distinct patient ids:", df_pat_s["patient_id"].nunique(), " rows:", len(df_pat_s))
print("dates of birth that look like a placeholder (1900):", df_pat_s["dob"].fillna("").str.startswith("1900").sum())

In [ ]:
# ----- the plan (wide: one column per month) -----
df_plan_s = pd.read_csv(RAW / "plan" / "plan_2026_H1.csv")
print(df_plan_s)
print("city names in the plan:", df_plan_s["city"].tolist())
plan_total_s = df_plan_s.drop(columns=["city", "metric"]).sum().sum()
print("plan total, all cities and months:", plan_total_s)

In [ ]:
# ----- results (one file: one row per test the lab ran) -----
result_files = sorted((RAW / "results").rglob("*.csv.gz"))
print("number of results files:", len(result_files))
df_res_s = pd.read_csv(result_files[0], dtype=str)
print(df_res_s.shape)
print(df_res_s.head())
print(df_res_s.isnull().sum())

## 5 · Id audit: does every table use the id format we expect?
A mismatch is not an error yet. It is a list of things to decide about in notebook 03.

In [ ]:
# patterns we expect (written as regular expressions)
legacy_pattern = r"^BK-[A-Z]{3}-\d{6}-\d{6}$"      # example BK-AHM-202501-000001
nova_pattern = r"^nova_[a-z]{3}_\d{7}$"            # example nova_che_0087556

# legacy booking id, before and after removing the spaces
raw_ids = df_leg["booking_id"]
share_raw = raw_ids.str.match(legacy_pattern).mean()
share_stripped = raw_ids.str.strip().str.match(legacy_pattern).mean()
print("legacy ids matching the pattern, as they arrive :", round(share_raw, 4))
print("legacy ids matching the pattern, after strip     :", round(share_stripped, 4))

# nova ids
share_nova = df_nova_small["booking_id"].str.match(nova_pattern).mean()
print("nova ids matching the pattern                    :", round(share_nova, 4))

# collection ids can be either type
share_coll = df_coll_s["booking_id"].str.match(legacy_pattern + "|" + nova_pattern).mean()
print("collections ids matching either pattern          :", round(share_coll, 4))
check("collections ids look like booking ids", share_coll > 0.99)

In [ ]:
# a small function that profiles any dataframe: type, missing rate, number of different values
def profile(df, name):
    rows = []
    for col in df.columns:
        rows.append({"table": name, "column": col, "dtype": str(df[col].dtype),
                     "null_rate": round(df[col].isnull().mean(), 4), "distinct": df[col].nunique()})
    return pd.DataFrame(rows)

df_profile = pd.concat([profile(df_leg, "bookings_legacy"), profile(df_nova_small, "bookings_nova"),
                        profile(df_coll_s, "collections"), profile(df_pat_s, "patients"),
                        profile(df_res_s, "results")], ignore_index=True)
print(df_profile.to_string())
df_profile.to_csv(CLEAN / "profile_columns.csv", index=False)

## 6 · SUM: what we learned (copy these into `docs/decisions-log.md`)

Fill the numbers from the cells above.

| What we found | Number | What we will do about it |
|---|---|---|
| Legacy files are latin-1 and the dates are day first | | read with the right encoding and an explicit date format |
| The header line is repeated inside some legacy files | | drop those rows and count them |
| Booking ids with spaces at the end | | strip all ids before any join |
| Grain: legacy is one row per test, nova is one object per booking | | flatten nova to one row per test |
| Nova keeps the old copy of a rescheduled visit | | drop the old copy, keep the replacement |
| Nova file month is not always the slot month | | always use `slot_at` for the period, never the file name |
| Blank test codes | | keep the row and flag it, do not guess a code |

## Before you move on
- Which table will we count tests from, and why not the other two? (approach-log row 1)
- Which join will be the hardest, and what did the id audit tell us about it?
- What did we expect the row counts to be before we looked? How far off were we?